<a href="https://colab.research.google.com/github/JuanbyGuada/Ciencia-de-Datos/blob/develop/Version_con_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLOQUE 1: Instalación de librerías necesarias


In [ ]:
!pip install -q pdfminer.six keybert sentence-transformers nltk google-generativeai

# BLOQUE 2: Imports y configuración


In [ ]:
import os
import re
import nltk
import google.generativeai as genai
from pdfminer.high_level import extract_text
from nltk.corpus import stopwords
from keybert import KeyBERT
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')
nltk.download('punkt')


# Colocá tu API Key de Google AI Studio
genai.configure(api_key="Poner_API_Key")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# BLOQUE 3: Función para extraer y limpiar texto de PDF


In [ ]:
def extract_and_clean_pdf(pdf_path):
    try:
        raw_text = extract_text(pdf_path)
        cleaned = re.sub(r'\n{2,}', '\n\n', raw_text)
        cleaned = re.sub(r'[^\w\sáéíóúÁÉÍÓÚñÑ.,;:¿?¡!()\-–]', '', cleaned)
        cleaned = re.sub(r'\s{2,}', ' ', cleaned)
        return cleaned
    except Exception as e:
        print(f"Error al procesar PDF: {e}")
        return ""

# BLOQUE 4: Extracción de conceptos clave con modelo BERT (KeyBERT)


In [ ]:
def extraer_conceptos_keybert(texto, top_n=10):
    try:
        stop_es = stopwords.words('spanish') + [
            "ejemplo", "etc", "tipo", "tipos", "puede", "diferentes", "dado",
            "si", "no", "como", "cada", "bien", "vez", "parte", "tres", "dos",
            "entonces", "años", "año", "miles", "decir", "lugar", "través", "muchos",
            "cual", "donde", "nuevo", "solo", "menos", "más", "gran", "base", "respecto",
            "fin", "inicio", "uso", "general", "ejemplos", "diferencia", "características"
        ]

        kw_model = KeyBERT(model="paraphrase-multilingual-MiniLM-L12-v2")

        vectorizer = CountVectorizer(
            ngram_range=(1, 10),
            stop_words=stop_es,
            max_features=10000
        )

        keywords = kw_model.extract_keywords(
            texto,
            vectorizer=vectorizer,
            top_n=top_n * 2,
            use_mmr=True,
            diversity=0.7
        )

        conceptos_filtrados = [kw[0] for kw in keywords if len(kw[0]) > 4 and kw[1] > 0.2]
        return conceptos_filtrados[:top_n]
    except Exception as e:
        print(f"Error en KeyBERT: {e}")
        return []

# BLOQUE 5: Generar preguntas tipo examen con Gemini

In [ ]:
def generar_preguntas_gemini(conceptos):
    prompt = f"""
Eres un experto en Ciencia de Datos. Genera 1 pregunta tipo examen MÚLTIPLE CHOICE en español, escalando la dificultad en cada una y aclarando el nivel de complejidad en porcentaje, utilizando estos conceptos clave: {', '.join(conceptos)}.
Cada pregunta debe tener 4 opciones (A, B, C, D) y aclarar cuál es la correcta al final con el siguiente formato:

> RESPUESTA: [Letra correcta]

Ejemplo:
Pregunta 1: ¿Qué es un análisis exploratorio de datos? (Nivel de Complejidad: 30%)
A) Técnica para redes neuronales
B) Enfoque para visualizar datos
C) Método de encriptación
D) Sistema de almacenamiento
> RESPUESTA: B

Ahora generá las preguntas:
"""
    try:
        modelo = genai.GenerativeModel("gemini-1.5-flash")
        respuesta = modelo.generate_content([prompt])
        return respuesta.text
    except Exception as e:
        print(f"Error con Gemini: {e}")
        return ""


# BLOQUE 6: Limpieza del texto generado por Gemini

In [ ]:
def limpiar_preguntas(texto):
    preguntas = re.split(r'(Pregunta \d+:)', texto)
    resultado = []
    for i in range(1, len(preguntas), 2):
        bloque = preguntas[i] + preguntas[i+1]
        if all(op in bloque for op in ['A)', 'B)', 'C)', 'D)']):
            resultado.append(bloque.strip())
    return "\n\n".join(resultado[:3]) if resultado else "No se generaron preguntas válidas"

# --- FLUJO PRINCIPAL ---


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Modelo multilingüe basado en BERT (compatible con español)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


def validar_preguntas_con_pdf(texto_pdf, preguntas_generadas, umbral=0.5):
    try:
        # Embedding del texto base (completo o resumen)
        embedding_texto = embedder.encode(texto_pdf, convert_to_tensor=True)

        # Dividir preguntas generadas
        bloques = re.findall(r'(Pregunta \d+:.*?(?:D\).+?))(?:\n\n|$)', preguntas_generadas, re.DOTALL)

        resultados = []

        for pregunta in bloques:
            pregunta_clean = re.sub(r'> *RESPUESTA:.*', '', pregunta).strip()
            emb_pregunta = embedder.encode(pregunta_clean, convert_to_tensor=True)
            score = util.cos_sim(embedding_texto, emb_pregunta).item()
            resultados.append((pregunta.strip(), round(score, 3)))

        return resultados
    except Exception as e:
        print(f"Error validando preguntas: {e}")
        return []


resultados = validar_preguntas_con_pdf(texto, preguntas_finales)

for i, (pregunta, score) in enumerate(resultados, 1):
    print(f"\n📌 Pregunta {i} (score: {score})\n{pregunta}")

#Interpretación del score
#🔵 0.7–1.0: Muy relacionada con el contenido del PDF.
#🟡 0.5–0.7: Aceptable, pero podría haber sido inventada o general.
#🔴 < 0.5: Probablemente inventada o irrelevante.

#Beneficios:
#Detección de alucinaciones: Identifica respuestas inventadas por el modelo
#Calidad garantizada: Asegura coherencia con el material fuente
#Retroalimentación útil: Proporciona scores para ajustar el prompt
#Contexto mejorado: Fragmentos relevantes mejoran la generación inicial


📌 Pregunta 1 (score: 0.497)
Pregunta 1: Un científico de datos está analizando datos de la MLBAM (Major League Baseball Advanced Media), incluyendo datos de audio (comentarios), imagen (jugadas), y vídeo (partidos completos).  Se le solicita construir un modelo que prediga la probabilidad de un jonrón en función de la velocidad de lanzamiento, la ubicación del lanzamiento y la trayectoria de la bola (obtenida a través de análisis de vídeo).  Considerando que la visualización de datos es crucial, pero que manejar múltiples gráficos superpuestos con los mismos nodos (jugadores) se vuelve confuso, ¿cuál de las siguientes afirmaciones acerca de los desafíos inherentes a este análisis es MÁS precisa? (Nivel de Complejidad: 60%)

A)  El desafío principal reside en la dificultad de almacenar y consultar los grandes volúmenes de datos de audio, imagen y vídeo, requiriendo herramientas técnicas de almacenamiento especializado y  consultas optimizadas, independientemente del modelo predictivo e

In [ ]:
pdf_name = "tipos_de_datos_-_ciencia_de_datos_0.pdf"  # Cambiá por el nombre real

if not os.path.exists(pdf_name):
    print(f"❌ No se encontró el archivo {pdf_name}")
else:
    print("✅ Procesando PDF...")
    texto = extract_and_clean_pdf(pdf_name)

    conceptos = extraer_conceptos_keybert(texto, top_n=10)
    print(f"🔑 Conceptos extraídos con BERT: {', '.join(conceptos)}")

    preguntas_raw = generar_preguntas_gemini(conceptos)
    preguntas_finales = limpiar_preguntas(preguntas_raw)

    print("\n📝 PREGUNTAS GENERADAS:\n")
    print(preguntas_finales)

✅ Procesando PDF...
🔑 Conceptos extraídos con BERT: datos tiende requerir herramientas técnicas principales categorías datos, confuso cualquier dato ser mostrado, dependen modelo, audio imagen vídeo datos, múltiples gráficos superpuestos mismos nodos imagina, natural generado, métricas específicas influencia persona camino corto personas, administrar consultar, almacenamiento, ser desafío computadoras mlbam major league baseball advanced

📝 PREGUNTAS GENERADAS:

Pregunta 1: Un científico de datos está analizando datos de la MLBAM (Major League Baseball Advanced Media), incluyendo datos de audio (comentarios), imagen (jugadas), y vídeo (partidos completos).  Se le solicita construir un modelo que prediga la probabilidad de un jonrón en función de la velocidad de lanzamiento, la ubicación del lanzamiento y la trayectoria de la bola (obtenida a través de análisis de vídeo).  Considerando que la visualización de datos es crucial, pero que manejar múltiples gráficos superpuestos con los mis